# 15. Date & Time Functions

*Module: 4 — Advanced querying · Companion summary: `lectures/15-date-and-time-functions.md` · SQL engine: SQLite (in-memory)*

## What you will study

- extract parts of a date (day, month, year)
- count rows per month / per year
- do date arithmetic and measure day differences
- know the dialect traps: DAY()/MONTH()/CURRENT DATE vs SQLite's strftime

---
## Setup — build the sample database this notebook runs on
*Run the cell below first: it creates an in-memory SQLite database and the `q()` helper that prints every query's result set.*

In [ ]:
import sqlite3
import textwrap

conn = sqlite3.connect(':memory:')  # throw-away in-memory database
conn.execute("PRAGMA foreign_keys = ON")

def q(sql):
    """Run a query, print formatted SQL, and display the result set as a table."""
    clean_sql = textwrap.dedent(sql).strip()

    # 1. Print formatted SQL query
    print("\n" + "=" * 60)
    print("QUERY:")
    print(textwrap.indent(clean_sql, "  "))
    print("-" * 60)

    cur = conn.execute(sql)
    rows = cur.fetchall()

    if cur.description:  # SELECT / statement returning rows
        headers = [d[0] for d in cur.description]
        str_rows = [[("" if v is None else str(v)) for v in r] for r in rows]

        # Calculate dynamic column widths for clean alignment
        widths = [
            max(len(h), *(len(r[i]) for r in str_rows), 1)
            for i, h in enumerate(headers)
        ] if str_rows else [len(h) for h in headers]

        header_line = " | ".join(h.ljust(widths[i]) for i, h in enumerate(headers))
        sep_line = "-+-".join("-" * widths[i] for i in range(len(headers)))

        print(header_line)
        print(sep_line)
        for r in str_rows:
            print(" | ".join(val.ljust(widths[i]) for i, val in enumerate(r)))
        if not str_rows:
            print("(0 rows returned)")
    else:  # DDL / DML (CREATE, INSERT, UPDATE, DELETE)
        affected = cur.rowcount
        print(f"Done (rows affected: {affected})" if affected != -1 else "Done.")

    print("=" * 60)

print('Connection ready:', conn)

## 🧪 Sample data — PETRESCUE again (dates this time)

In [ ]:
q("CREATE TABLE petrescue (id INTEGER PRIMARY KEY, animal TEXT, quantity INTEGER, cost REAL, rescue_date TEXT)")
q("""
INSERT INTO petrescue VALUES
    (1, 'Dog',   1, 100.0, '2018-05-01'),
    (2, 'Cat',   1,  50.0, '2018-05-10'),
    (3, 'Dog',   2, 120.0, '2018-06-15'),
    (4, 'Bird',  3,  60.0, '2019-05-02'),
    (5, 'Cat',   2,  80.0, '2019-06-20'),
    (6, 'Dog',   1, 150.0, '2020-01-05'),
    (7, 'Bird',  5,  75.0, '2020-02-14'),
    (8, 'Cat',   1,  45.0, '2020-05-30')
""")
q("SELECT * FROM petrescue")

## 🧪 Extract parts of a date

📕 The course uses DB2: `DAY(rescue_date)`, `MONTH(rescue_date)`, `YEAR(rescue_date)`.
SQLite spells the same idea with **strftime**:

In [ ]:
q("SELECT rescue_date, strftime('%d', rescue_date) AS day, strftime('%m', rescue_date) AS month, strftime('%Y', rescue_date) AS year FROM petrescue")

## 🧪 Count rescues in a given month

In [ ]:
q("SELECT COUNT(*) FROM petrescue WHERE strftime('%m', rescue_date) = '05'")
q("SELECT COUNT(*) FROM petrescue WHERE strftime('%Y', rescue_date) = '2020'")

## 🧪 Date arithmetic — DATE(date, modifier)

In [ ]:
q("SELECT id, rescue_date, DATE(rescue_date, '+3 day') AS three_days_later FROM petrescue LIMIT 3")
q("SELECT id, rescue_date, DATE(rescue_date, '-1 month') AS week_before FROM petrescue LIMIT 3")

## 🧪 Day difference with JULIANDAY

In [ ]:
q("SELECT id, rescue_date, CAST(julianday('2020-12-31') - julianday(rescue_date) AS INTEGER) AS days_to_year_end FROM petrescue LIMIT 4")

## 🧪 Today

In [ ]:
q("SELECT date('now') AS today")

## 💡 Tips & tricks

- SQLite stores dates best as **TEXT in ISO format** (`YYYY-MM-DD`); lexicographic
  order = chronological order, so comparisons and sorting just work.
- 📕 Dialect cheat sheet:
  | Idea | DB2 (course) | SQLite | MySQL |
  |---|---|---|---|
  | month part | `MONTH(d)` | `strftime('%m', d)` | `MONTH(d)` |
  | add 3 days | `d + 3 DAYS` | `DATE(d, '+3 day')` | `DATE_ADD(d, INTERVAL 3 DAY)` |
  | today | `CURRENT DATE` | `date('now')` | `CURRENT_DATE` |

## 🎯 Your turn

1. How many rescues happened in May of 2019?
2. Each rescue's date shifted back 10 days.
3. Number of years since the oldest rescue.

In [ ]:
# 💡 Solution
q("SELECT COUNT(*) FROM petrescue WHERE strftime('%m', rescue_date) = '05' AND strftime('%Y', rescue_date) = '2019'")
q("SELECT id, rescue_date, DATE(rescue_date, '-10 day') FROM petrescue LIMIT 4")
q("SELECT (strftime('%Y', 'now') - strftime('%Y', (SELECT MIN(rescue_date) FROM petrescue))) AS years_since")